# 10. MDIS 원자료 전처리

- 입력: `data/processed/mdis_regional_B.csv` (09 노트북이 만든 통합 원자료)
- 출력: `data/processed/mdis_clean.csv` (응답자 1명이 1행, 분석에 쓰는 열만)

두 파일 모두 개인 단위 원자료라 저장소에 올리지 않는다. 열 설명은 `docs/ai_exposure_design_eda.md` 8-8절에 있다.

하는 일은 세 가지다.

1. 분석에 쓰는 열만 남기고 영어 이름으로 바꾼다.
2. 15~59세만 남긴다(청년 15~29세와 비교군 30~59세).
3. 파생 변수(연령군, 직업 중분류, 근속 개월 수, 신규 진입 여부)를 만든다.

**하지 않는 것**: 중복 행 제거. 공공용 원자료에는 개인 식별번호가 없어서, 모든 값이 같은 행도 서로 다른 사람이다. 지우면 가중 합계가 공표값과 어긋난다.

In [ ]:
import sys

sys.path.append("..")  # 노트북 폴더 기준으로 src를 찾는다 (README 6절)
import numpy as np
import pandas as pd

from src import config

SRC = config.DATA_PROCESSED / "mdis_regional_B.csv"
OUT = config.DATA_PROCESSED / "mdis_clean.csv"

KEEP = {  # 원래 이름 → 새 이름
    "period": "period",
    "조사연월": "survey_ym",
    "성별코드": "sex",
    "만연령": "age",
    "교육정도변환코드": "edu3",
    "경제활동인구상태코드": "lf_status",
    "활동상태코드": "activity",
    "3자리_8차직업소분류코드": "occ3",
    "3자리_11차산업소분류코드": "ind3",
    "종사상지위코드": "emp_status",
    "조사대상주간_직장시작연월": "start_ym",
    "고용계약기간정함여부": "contract_fixed",
    "최근3개월_월평균급여": "wage",
    "조사대상주간_근무총계시간수": "hours",
    "직장종사자규모코드": "firm_size",
    "이직횟수코드": "job_changes",
    "시도전국가중값": "weight",
}
CODES = ["period", "sex", "edu3", "lf_status", "activity", "occ3", "ind3", "emp_status", "contract_fixed", "firm_size", "job_changes"]

## 1. 필요한 열만 읽기

코드 열은 글자로 읽는다(`011` 같은 코드의 앞자리 0이 사라지지 않도록).

In [ ]:
header = pd.read_csv(SRC, nrows=0).columns
df = pd.read_csv(SRC, usecols=list(KEEP), dtype={k: str for k, v in KEEP.items() if v in CODES}).rename(columns=KEEP)[list(KEEP.values())]
n_raw = len(df)
print(f"열 {len(header)}개 → {df.shape[1]}개, 행 {n_raw:,}")
print("뺀 열:", ", ".join(c for c in header if c not in KEEP))

## 2. 15~59세만 남기기

취업자뿐 아니라 실업자와 비경제활동인구도 남긴다(고용률의 분모가 필요하기 때문).

In [ ]:
df = df[df.age.between(15, 59)].copy()
df["age_group"] = np.where(df.age <= 29, "y15_29", "a30_59")
print(f"행 {n_raw:,} → {len(df):,} (60세 이상 {n_raw - len(df):,}행 제외)")

## 3. 파생 변수

- `employed`: 취업자 여부 (`lf_status` = 1)
- `occ2`: 직업 중분류(소분류 코드 앞 두 자리). `panel_analysis.csv`의 `code`와 같다.
- `tenure_m`: 현 직장 근속 개월 수 = 조사연월 − 직장 시작 연월. 시작 연월이 형식에 맞지 않으면 빈칸.
- `new_entrant`: 취업자 중 현 직장 12개월 미만. 첫 취업과 이직이 모두 들어간다.

In [ ]:
df["employed"] = df.lf_status == "1"
df["occ2"] = df.occ3.str[:2]

start_y, start_m = df.start_ym // 100, df.start_ym % 100
valid = start_m.between(1, 12) & start_y.between(1930, df.survey_ym // 100)
df["tenure_m"] = ((df.survey_ym // 100 - start_y) * 12 + (df.survey_ym % 100 - start_m)).where(valid)
df["new_entrant"] = df.employed & (df.tenure_m < 12)
n_bad = int((df.employed & ~valid).sum())
print(f"직장 시작 연월이 형식에 맞지 않는 취업자 {n_bad}명 → tenure_m 빈칸, new_entrant는 False")
assert not (df.tenure_m < 0).any(), "근속이 음수인 행이 있음"

## 4. 빈칸은 그대로 둔다

남은 빈칸은 오류가 아니라 "해당 없음"이다. 0이나 평균으로 채우지 않는다.

- 직업·산업·종사상지위·근속: 취업자가 아니면 빈칸
- `wage`, `contract_fixed`: 임금근로자(`emp_status` 1~3)가 아니면 빈칸
- `activity`: 취업자면 빈칸
- `firm_size`: 2021년부터 조사, `job_changes`: 하반기에만 조사

In [ ]:
emp = df[df.employed]
na = pd.DataFrame({"전체 빈칸%": df.isna().mean() * 100, "취업자 빈칸%": emp.isna().mean() * 100}).round(1)
print(na[na["전체 빈칸%"] > 0].to_string())
wage_workers = emp.emp_status.isin(["1", "2", "3"])
assert emp.loc[wage_workers, "wage"].notna().all() and emp.loc[~wage_workers, "wage"].isna().all(), "급여 빈칸이 임금근로자 여부와 맞지 않음"
assert emp[["occ3", "ind3", "emp_status"]].notna().all().all(), "취업자의 직업·산업·지위에 빈칸이 있음"
print("급여(만원) 분위:", emp.wage.quantile([0.01, 0.5, 0.99, 1]).round(0).to_dict(), "← 최댓값은 이상값으로 보이므로 평균 대신 중위수를 쓴다")

## 5. 검사 후 저장

가중값을 더한 청년 취업자가 반기마다 KOSIS 공표값과 1% 안에서 맞아야 하고, 직업 중분류 코드가 분석 패널의 56개 안에 있어야 한다.

In [ ]:
panel = pd.read_csv(config.DATA_PROCESSED / "panel_mid.csv", dtype={"code": str, "period": str})
youth = df[df.employed & (df.age_group == "y15_29")]
cmp = pd.concat([youth.groupby("period").weight.sum().div(1000).rename("원자료(천명)"),
                 panel.groupby("period").y15_29.sum().rename("KOSIS(천명)")], axis=1).dropna()
cmp["차이(%)"] = 100 * (cmp["원자료(천명)"] / cmp["KOSIS(천명)"] - 1)
print(cmp.round(2).to_string())
assert cmp["차이(%)"].abs().max() < 1, "KOSIS 공표값과 1% 넘게 다름"
assert set(emp.occ2) <= set(panel.code), "분석 패널에 없는 직업 중분류 코드가 있음"
assert df.period.nunique() == len(cmp), "반기 수가 맞지 않음"

COLS = ["period", "sex", "age", "age_group", "edu3", "lf_status", "employed", "activity", "occ3", "occ2", "ind3", "emp_status",
        "start_ym", "tenure_m", "new_entrant", "contract_fixed", "wage", "hours", "firm_size", "job_changes", "weight"]
for c in ("start_ym", "tenure_m", "wage"):
    df[c] = df[c].astype("Int64")   # 빈칸이 있는 정수 열이 200.0처럼 저장되지 않도록 (hours는 40.5 같은 값이 있어 그대로 둠)
df[COLS].to_csv(OUT, index=False, encoding="utf-8")
print(f"저장: {OUT.name}, {len(df):,}행 × {len(COLS)}열, {OUT.stat().st_size / 1e6:.0f}MB")